# Single-omics analysis

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.cluster import AgglomerativeClustering, SpectralClustering
from sklearn.metrics import silhouette_score, adjusted_mutual_info_score
from sklearn.preprocessing import StandardScaler
import ast
import itertools
from scipy.stats import kruskal
from ptitprince import PtitPrince as pt
from scipy.stats import hmean
import matplotlib.lines as mlines

In [ ]:
from matplotlib import rcParams
sns.set_theme(style='ticks')
rcParams.update({
    'font.size': 11,
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica'],
    'axes.labelsize': 11,
    'axes.titlesize': 11,
    'axes.edgecolor': 'black',
    'axes.linewidth': 0.8,
    'xtick.labelsize': 11,
    'ytick.labelsize': 11,
    'xtick.direction': 'out',
    'ytick.direction': 'out',
    'xtick.major.size': 3,
    'ytick.major.size': 3,
    'legend.fontsize': 10,
    'legend.frameon': False,
    'savefig.format': 'svg',
    'savefig.dpi': 300,  # Still useful for rasterized elements
    'figure.dpi': 100,
    'figure.figsize': (3.5, 2.5),  # Approx. half-column width
    'figure.constrained_layout.use': True,
    'svg.fonttype': 'none',  # Keep text as editable text (not paths)
    'axes.spines.top': False,
    'axes.spines.right': False,
})
colorblind_palette = sns.color_palette('colorblind')

In [ ]:
# Get patient sampling used in experiments
sampling = pd.read_csv("../results/cluster_analysis/benchmarking_files/firstbench_2clusters.csv")
patients = list(sampling.iloc[:10, sampling.columns.get_loc("y_pred_idx")])

In [ ]:
# Collect preprocessed data (run data_preprocessing.py script for samples with all modalities). Name dfs (useful for later)
methyl_data = pd.read_csv("../data/TCGA/omics_data/preprocessed/patients_with_all_views/TCGA_PDAC_subset_Methylation.csv", index_col=0)
methyl_data.name = "Methylation"
cna_data = pd.read_csv("../data/TCGA/omics_data/preprocessed/patients_with_all_views/TCGA_PDAC_subset_CNA.csv", index_col=0)
cna_data.name = "CNA"
mirna_data = pd.read_csv("../data/TCGA/omics_data/preprocessed/patients_with_all_views/TCGA_PDAC_subset_miRNA.csv", index_col=0)
mirna_data.name = "miRNA"
mutation_data = pd.read_csv("../data/TCGA/omics_data/preprocessed/patients_with_all_views/TCGA_PDAC_subset_Mutation.csv", index_col=0)
mutation_data.name = "Mutation"
rnaseq_data = pd.read_csv("../data/TCGA/omics_data/preprocessed/patients_with_all_views/TCGA_PDAC_subset_RNAseq.csv", index_col=0)
rnaseq_data.name = "RNAseq"
rppa_data = pd.read_csv("../data/TCGA/omics_data/preprocessed/patients_with_all_views/TCGA_PDAC_subset_RPPA.csv", index_col=0)
rppa_data.name = "RPPA"

In [ ]:
# Parameters for clustering
modalities = [methyl_data, cna_data, mirna_data, mutation_data, rnaseq_data, rppa_data]
clusters = [2, 3, 4, 5]
algorithms = ["Hierarchical", "Spectral"]
RANDOM_STATE = 42

In [ ]:
# Perform clustering
import warnings
warnings.filterwarnings('ignore')
rows = []
for algorithm in algorithms:
    for modality in modalities:
        for cluster in clusters:
            run_counter = 0
            for subset in patients:
                subset_patients = ast.literal_eval(subset)
                subset_data = modality.loc[subset_patients]
                subset_array = subset_data.to_numpy()
                scaled_df = StandardScaler().fit_transform(subset_array)
                if algorithm == "Hierarchical":
                    labels = AgglomerativeClustering(n_clusters=cluster).fit_predict(scaled_df)
                    silhouette = silhouette_score(X=scaled_df, labels=labels, random_state=RANDOM_STATE)
                elif algorithm == "Spectral":
                    labels = SpectralClustering(n_clusters=cluster, assign_labels="cluster_qr", random_state=RANDOM_STATE).fit_predict(scaled_df)
                    silhouette = silhouette_score(X=scaled_df, labels=labels, random_state=RANDOM_STATE)
                rows.append({"modalities":modality.name, "algorithm":algorithm, "n_clusters":cluster, "run_n":run_counter, "n_samples":len(subset_patients), "y_pred":labels, "y_pred_idx":subset_patients, "silhouette":silhouette})
                run_counter += 1

In [ ]:
singleomics_df = pd.DataFrame(rows)
singleomics_df.replace({"modalities": {"CNA": "100000", "Methylation": "010000", "Mutation": "001000", "RNAseq": "000100", "RPPA": "000010", "miRNA": "000001"}}, inplace=True)
singleomics_df["dataset"] = "Baseline"
singleomics_df["cluster_sizes"] = singleomics_df["y_pred"].apply(lambda x: pd.Series(x).value_counts(dropna=False).to_dict())
singleomics_df["relative_cluster_sizes"] = singleomics_df["y_pred"].apply(lambda x: pd.Series(x).value_counts(normalize=True, dropna=False).to_dict())

<br>

In [ ]:
# Get data from previous benchmarks to compare
bench1_2clusters = pd.read_csv('../results/cluster_analysis/benchmarking_files/firstbench_2clusters.csv',
                               dtype={'view_combination': str},
                               converters={'y_pred': eval, 'y_pred_idx': eval, 
                                           'relative_cluster_sizes': lambda x: eval(x.replace(': ', ':'))})
bench1_3clusters = pd.read_csv('../results/cluster_analysis/benchmarking_files/firstbench_3clusters.csv',
                               dtype={'view_combination': str},
                               converters={'y_pred': eval, 'y_pred_idx': eval, 
                                           'relative_cluster_sizes': lambda x: eval(x.replace(': ', ':'))})
bench1_4clusters = pd.read_csv('../results/cluster_analysis/benchmarking_files/firstbench_4clusters.csv',
                               dtype={'view_combination': str},
                               converters={'y_pred': eval, 'y_pred_idx': eval, 
                                           'relative_cluster_sizes': lambda x: eval(x.replace(': ', ':'))})
bench1_5clusters = pd.read_csv('../results/cluster_analysis/benchmarking_files/firstbench_5clusters.csv',
                               dtype={'view_combination': str},
                               converters={'y_pred': eval, 'y_pred_idx': eval, 
                                           'relative_cluster_sizes': lambda x: eval(x.replace(': ', ':'))})
frames = [bench1_2clusters, bench1_3clusters, bench1_4clusters, bench1_5clusters]
bench1_file = pd.concat(frames)
bench1_df = bench1_file[["dataset", "view_combination", "algorithm", "n_clusters", "run_n", "n_samples", "y_pred", "y_pred_idx", "silhouette", "cluster_sizes", "relative_cluster_sizes"]]
bench1_df.rename(columns={"view_combination":"modalities"}, inplace=True)

Functions (edited to work with this script)

In [ ]:
def remove_small_clusters(df: pd.DataFrame, n_outliers: int, verbose: bool):
    valid_results = df[df['relative_cluster_sizes'].apply(lambda d: all(value >= 0.1 for value in d.values()))]
    outlier_results = df[df['relative_cluster_sizes'].apply(lambda f: any(value < 0.1 for value in f.values()))]
    outlier_patients = {}
    for index, row in outlier_results.iterrows():
        cluster_sizes = row['relative_cluster_sizes']
        small_clusters =  [cluster for cluster, size in cluster_sizes.items() if size < 0.1]
        patients = row['y_pred_idx']
        clusters = row['y_pred']
        for patient, cluster in zip(patients, clusters):
            if cluster in small_clusters:
                if patient not in outlier_patients:
                    outlier_patients[patient] = 1
                else:
                    outlier_patients[patient] += 1
    for key in outlier_patients:
        outlier_patients[key] /= len(outlier_results)
    top_outliers = sorted(outlier_patients.items(), key=lambda x: x[1], reverse=True)[:n_outliers]
    df_top_outliers = pd.DataFrame(data=top_outliers, columns=['Patient ID', 'Count'])
    top_outlier_patients = [patient for patient, count in top_outliers]
    if verbose == True:
        print(f"Top {n_outliers} outlier patients: {top_outlier_patients}")
    return valid_results, outlier_results, df_top_outliers

In [ ]:
from tqdm import tqdm
import itertools

def calc_ami(results: pd.DataFrame, random_state=None, progress_bar=True):
    results["sorted_y_pred_idx"] = results["y_pred_idx"].apply(sorted)
    results["sorted_y_pred"] = results.apply(
        lambda row: [row["y_pred"][row["y_pred_idx"].index(patient_id)] for patient_id in row["sorted_y_pred_idx"]],
        axis=1)
    base_columns = ['dataset', 'modalities', 'algorithm', 'n_clusters', 'run_n', 'sorted_y_pred', 'sorted_y_pred_idx', 'silhouette']
    normalised_columns = [col for col in results.columns if 'normalised' in col]
    all_columns = base_columns + normalised_columns
    alg_stability = results[all_columns]
    alg_uns_metrics = alg_stability.drop(columns=['sorted_y_pred', 'sorted_y_pred_idx', 'run_n'])
    # Group by taking mean of metrics
    alg_uns_metrics = alg_uns_metrics.groupby(["dataset", "algorithm", "modalities", "n_clusters"], as_index=False).mean()
    iterator = alg_stability["dataset"].unique()
    if progress_bar:
        iterator = tqdm(iterator)
    for dataset in iterator:
        preds_dataset = alg_stability.loc[
            (alg_stability["dataset"] == dataset), ["algorithm", 'n_clusters', 'modalities', "run_n", "sorted_y_pred", "sorted_y_pred_idx"]]
        for alg in preds_dataset["algorithm"].unique():
            pred_alg = preds_dataset[preds_dataset["algorithm"] == alg]
            for view in pred_alg["modalities"].unique():
                pred_alg_view = pred_alg[
                    pred_alg["modalities"] == view]
                for cluster in pred_alg_view["n_clusters"].unique():
                    pred_alg_view_clus = pred_alg_view[
                        pred_alg_view['n_clusters'] == cluster]
                    amis = []
                    for run_1, run_2 in set(itertools.combinations(pred_alg_view_clus["run_n"].unique(), 2)):
                        pred1_alg = pred_alg_view_clus.loc[(pred_alg_view_clus["run_n"] == run_1), "sorted_y_pred"].to_list()[0]
                        pred2_alg = pred_alg_view_clus.loc[(pred_alg_view_clus["run_n"] == run_2), "sorted_y_pred"].to_list()[0]
                        pred1_idx = pred_alg_view_clus.loc[(pred_alg_view_clus["run_n"] == run_1), "sorted_y_pred_idx"].to_list()[0]
                        pred2_idx = pred_alg_view_clus.loc[(pred_alg_view_clus["run_n"] == run_2), "sorted_y_pred_idx"].to_list()[0]
                        # Only select samples in common for stability metrics
                        common_samples = list(set(pred1_idx) & set(pred2_idx))
                        pred1_common = [pred1_alg[pred1_idx.index(i)] for i in common_samples]
                        pred2_common = [pred2_alg[pred2_idx.index(i)] for i in common_samples]
                        amis.append(adjusted_mutual_info_score(pred1_common, pred2_common))
                    alg_uns_metrics.loc[(alg_uns_metrics["dataset"] == dataset) &
                                        (alg_uns_metrics["algorithm"] == alg) & 
                                        (alg_uns_metrics["modalities"] == view) & 
                                        (alg_uns_metrics["n_clusters"] == cluster),
                    ["AMI"]] = np.mean(amis)
    return alg_uns_metrics

In [ ]:
# Function to normalise metrics (adapted so that it can skip nan values but still follow normalisation process)
def add_normalised_metric(df, variable_to_normalise, metric, max_score, greater_is_better=True):
    possible_variables = ["dataset", "algorithm", "modalities", "n_clusters"]
    valid_variables = [var for var in possible_variables if var != variable_to_normalise]
    df_new = df.copy()
    def recursive_loop(subset, remaining_vars, current_filters):
        if not remaining_vars:
            scores = subset[metric].values
            max_value = max_score if max_score else scores.max()
            relative_score = scores / max_value
            if not greater_is_better:
                relative_score = 1 - relative_score
            condition = True
            for key, value in current_filters.items():
                condition &= (df_new[key] == value)
            df_new.loc[condition, f'normalised_{metric}'] = relative_score
            return
        current_var = remaining_vars[0]
        for unique_value in subset[current_var].unique():
            filtered_subset = subset[subset[current_var] == unique_value]
            recursive_loop(filtered_subset, remaining_vars[1:], {**current_filters, current_var: unique_value})
    df_new[f'normalised_{metric}'] = float('nan')
    recursive_loop(df_new, valid_variables, {})
    return df_new

<br>

Create dfs for metrics normalised wrt modality / view combinations.

In [ ]:
# Perform analysis and normalisation on original dataset from first benchmark. 
import warnings
warnings.filterwarnings("ignore")
bench1_valid, bench1_outliers, bench1_outlier_patients = remove_small_clusters(bench1_df, 20, verbose = False)
bench1_valid["silhouette"] = bench1_valid["silhouette"].clip(lower=0)
bench1_mod_sil = add_normalised_metric(bench1_valid, variable_to_normalise='modalities', metric='silhouette', max_score= False, greater_is_better=True)
bench1_mod_ami = calc_ami(bench1_mod_sil, random_state=42, progress_bar=True)
bench1_mod_ami['AMI'] = bench1_mod_ami['AMI'].fillna(value=0)
bench1_mod_ami['AMI'] = bench1_mod_ami['AMI'].clip(lower=0)
modality_multi = add_normalised_metric(bench1_mod_ami, variable_to_normalise='modalities', metric='AMI', max_score= False, greater_is_better=True)
modality_multi['n_views'] = modality_multi['modalities'].str.count('1')
modality_multi['gps'] = modality_multi[['normalised_silhouette', 'normalised_AMI']].apply(lambda x: hmean(x), axis=1)
modality_multi.sort_values('gps', ascending=False, inplace=True)
modality_multi

In [ ]:
# Perform analysis and normalisation on baseline dataset
import warnings
warnings.filterwarnings("ignore")
baseline_valid, baseline_outliers, baseline_outlier_patients = remove_small_clusters(singleomics_df, 20, verbose = False)
baseline_valid["silhouette"] = baseline_valid["silhouette"].clip(lower=0)
baseline_mod_sil = add_normalised_metric(baseline_valid, variable_to_normalise='modalities', metric='silhouette', max_score=bench1_valid["silhouette"].max(), greater_is_better=True)
baseline_mod_ami = calc_ami(baseline_mod_sil, random_state=42, progress_bar=True)
baseline_mod_ami['AMI'] = baseline_mod_ami['AMI'].fillna(value=0)
baseline_mod_ami['AMI'] = baseline_mod_ami['AMI'].clip(lower=0)
modality_base = add_normalised_metric(baseline_mod_ami, variable_to_normalise='modalities', metric='AMI', max_score=bench1_mod_ami["AMI"].max(), greater_is_better=True)
modality_base['n_views'] = modality_base['modalities'].str.count('1')
modality_base['gps'] = modality_base[['normalised_silhouette', 'normalised_AMI']].apply(lambda x: hmean(x), axis=1)
modality_base.sort_values('gps', ascending=False, inplace=True)
modality_base

In [ ]:
modality_df = pd.concat([modality_base, modality_multi])

First, check position across all modalities where single-omics lie

In [ ]:
metric = 'gps'
views = ['CNA', 'Methyl', 'Mutations', 'RNAseq', 'RPPA', 'miRNA']
results1 = modality_df.copy()

combinations_average = results1.groupby(['modalities']).mean(numeric_only=True).sort_values(by=metric, ascending=False)
combinations_sorted = combinations_average.index.tolist()

fig, ax = plt.subplots(4, 1, sharex=True, figsize=(20,7), height_ratios=[0.5, 0.3, 0.1, 0.1])

# First plot: boxplots with combined metric score for each combination
sns.boxplot(data=results1, x='modalities', y=metric, ax=ax[0], 
            order=combinations_sorted, width=0.7, color='white', showmeans=True, 
            meanprops={"marker": "^", "markerfacecolor": "green", "markeredgecolor": "green"})
mean_legend = mlines.Line2D([], [], color='green', marker='^', linestyle='None', markersize=8, label='Mean')
# ax[0].legend(handles=[mean_legend], loc='best')
ax[0].set_ylabel('General performance score')
ax[0].set_xlabel('')
ax[0].set_axisbelow(True)
ax[0].set_ylim(-0.05, 1.05)
for line in ax[0].lines:
    line.set_color('black')
    line.set_xdata(line.get_xdata() + 0.5)
for patch in ax[0].patches:
    patch.set_edgecolor('black')
    vertices = patch.get_path().vertices
    vertices[:, 0] += 0.5

# Second plot: heatmap showing modalities present 
views_matrix = combinations_average.reset_index()
views_matrix_expanded = views_matrix['modalities'].apply(lambda x: pd.Series(list(x))).astype(int)
views_matrix_expanded.columns = views
views_matrix_expanded.index = views_matrix['modalities']
views_ordered = ['Methyl', 'miRNA', 'CNA', 'RNAseq', 'Mutations', 'RPPA'] 
# (^this is a bit of cheating, it is the order from highest to lowest of the modalities as calculated in the next step)
views_matrix_expanded = views_matrix_expanded.reindex(columns=views_ordered)
sns.heatmap(views_matrix_expanded.T, cmap='Blues', linewidths=0.1, linecolor='black', 
            cbar=False, ax=ax[1]).set(xlabel=None)
# ax[1].set_ylabel('Modalities')
ax[1].tick_params(axis='x', bottom=True, labelbottom=False)

# Third plot: heatmap showing number of modalities
unique_nviews_data = combinations_average['n_views'].to_frame()
sns.heatmap(unique_nviews_data.T, cmap='Reds', linewidths=0.1, linecolor='black', square=True,
            cbar=True, cbar_kws=dict(use_gridspec=True, location="bottom", pad=0.2), ax=ax[2], 
            yticklabels='', xticklabels=unique_nviews_data.index).set(xlabel=None)
ax[2].set_ylabel('Number of \nmodalities', labelpad=30, rotation=0, va='center')

# Fourth plot: heatmap showing number of features
features = [2185, 385, 52, 1419, 71, 192]   # From previous step, in same order
feature_sums = {}
for combination in combinations_sorted:
    total = sum(features[i] for i, bit in enumerate(combination) if bit == '1')
    feature_sums[combination] = total
features_df = pd.DataFrame(feature_sums, index=['number of features'])
sns.heatmap(features_df, cmap='Oranges', linewidths=0.1, linecolor='black', cbar=True, 
            cbar_kws=dict(use_gridspec=True, location="bottom", ticks=[123, 2150, 4304], pad=0.2), ax=ax[3], square=True,
            yticklabels='', xticklabels=features_df.columns).set(xlabel=None)
ax[3].set_ylabel('Number of \nfeatures', labelpad=30, rotation=0, va='center')
ax[3].set_xticklabels('')

# plt.tight_layout()
plt.savefig('figures/combinations.svg', bbox_inches='tight')
plt.show()

In [ ]:
# DF normalised with respect to algorithm
import warnings
warnings.filterwarnings("ignore")
bench1_valid, bench1_outliers, bench1_outlier_patients = remove_small_clusters(bench1_df, 20, verbose = False)
bench1_valid["silhouette"] = bench1_valid["silhouette"].clip(lower=0)
bench1_alg_sil = add_normalised_metric(bench1_valid, variable_to_normalise='algorithm', metric='silhouette', max_score= False, greater_is_better=True)
bench1_alg_ami = calc_ami(bench1_alg_sil, random_state=42, progress_bar=True)
bench1_alg_ami['AMI'] = bench1_alg_ami['AMI'].fillna(value=0)
bench1_alg_ami['AMI'] = bench1_alg_ami['AMI'].clip(lower=0)
algs_multi = add_normalised_metric(bench1_alg_ami, variable_to_normalise='algorithm', metric='AMI', max_score= False, greater_is_better=True)
algs_multi['n_views'] = algs_multi['modalities'].str.count('1')
algs_multi['gps'] = algs_multi[['normalised_silhouette', 'normalised_AMI']].apply(lambda x: hmean(x), axis=1)
algs_multi.sort_values('gps', ascending=False, inplace=True)

baseline_valid, baseline_outliers, baseline_outlier_patients = remove_small_clusters(singleomics_df, 20, verbose = False)
baseline_valid["silhouette"] = baseline_valid["silhouette"].clip(lower=0)
baseline_alg_sil = add_normalised_metric(baseline_valid, variable_to_normalise='algorithm', metric='silhouette', max_score=bench1_valid["silhouette"].max(), greater_is_better=True)
baseline_alg_ami = calc_ami(baseline_alg_sil, random_state=42, progress_bar=True)
baseline_alg_ami['AMI'] = baseline_alg_ami['AMI'].fillna(value=0)
baseline_alg_ami['AMI'] = baseline_alg_ami['AMI'].clip(lower=0)
algs_base = add_normalised_metric(baseline_alg_ami, variable_to_normalise='algorithm', metric='AMI', max_score=bench1_alg_ami["AMI"].max(), greater_is_better=True)
algs_base['n_views'] = algs_base['modalities'].str.count('1')
algs_base['gps'] = algs_base[['normalised_silhouette', 'normalised_AMI']].apply(lambda x: hmean(x), axis=1)
algs_base.sort_values('gps', ascending=False, inplace=True)

algs_df = pd.concat([algs_base, algs_multi])

In [ ]:
# DF normalised with respect to no. clusters
import warnings
warnings.filterwarnings("ignore")
bench1_valid, bench1_outliers, bench1_outlier_patients = remove_small_clusters(bench1_df, 20, verbose = False)
bench1_valid["silhouette"] = bench1_valid["silhouette"].clip(lower=0)
bench1_cluster_sil = add_normalised_metric(bench1_valid, variable_to_normalise='n_clusters', metric='silhouette', max_score= False, greater_is_better=True)
bench1_cluster_ami = calc_ami(bench1_cluster_sil, random_state=42, progress_bar=True)
bench1_cluster_ami['AMI'] = bench1_cluster_ami['AMI'].fillna(value=0)
bench1_cluster_ami['AMI'] = bench1_cluster_ami['AMI'].clip(lower=0)
clusters_multi = add_normalised_metric(bench1_cluster_ami, variable_to_normalise='n_clusters', metric='AMI', max_score= False, greater_is_better=True)
clusters_multi['n_views'] = clusters_multi['modalities'].str.count('1')
clusters_multi['gps'] = clusters_multi[['normalised_silhouette', 'normalised_AMI']].apply(lambda x: hmean(x), axis=1)
clusters_multi.sort_values('gps', ascending=False, inplace=True)

baseline_valid, baseline_outliers, baseline_outlier_patients = remove_small_clusters(singleomics_df, 20, verbose = False)
baseline_valid["silhouette"] = baseline_valid["silhouette"].clip(lower=0)
baseline_clusters_sil = add_normalised_metric(baseline_valid, variable_to_normalise='n_clusters', metric='silhouette', max_score=bench1_valid["silhouette"].max(), greater_is_better=True)
baseline_clusters_ami = calc_ami(baseline_clusters_sil, random_state=42, progress_bar=True)
baseline_clusters_ami['AMI'] = baseline_clusters_ami['AMI'].fillna(value=0)
baseline_clusters_ami['AMI'] = baseline_clusters_ami['AMI'].clip(lower=0)
clusters_base = add_normalised_metric(baseline_clusters_ami, variable_to_normalise='n_clusters', metric='AMI', max_score=bench1_cluster_ami["AMI"].max(), greater_is_better=True)
clusters_base['n_views'] = clusters_base['modalities'].str.count('1')
clusters_base['gps'] = clusters_base[['normalised_silhouette', 'normalised_AMI']].apply(lambda x: hmean(x), axis=1)
clusters_base.sort_values('gps', ascending=False, inplace=True)

clusters_df = pd.concat([clusters_base, clusters_multi])

In [ ]:
def raincloud_plots_variables(df, column_name, metric_name, ax, ylabel):
    mean_values = df.groupby(column_name)[metric_name].mean().sort_values(ascending=False)
    sorted_categories = mean_values.index.tolist()
    metric_subsets = [df[df[column_name] == cat][metric_name].values for cat in sorted_categories]
    pt.RainCloud(x=column_name, y=metric_name, data=df, bw=0.2, palette=[sns.color_palette('colorblind')[0]],
                 width_viol=0.4, ax=ax, orient="v", move=0.2, order=sorted_categories, alpha=0.8)
    means = df.groupby(column_name)[metric_name].mean().loc[sorted_categories]
    sns.scatterplot(x=range(len(sorted_categories)), y=means.values, ax=ax, color=sns.color_palette('colorblind')[2], s=50, marker='^', zorder=10)
    ax.set_xticks(range(len(sorted_categories)))
    ax.set_xticklabels(sorted_categories)
    ax.set_ylabel(ylabel)
    ax.set_ylim(-0.05, 1.05)
    ax.set_axisbelow(True)
    xticks = plt.xticks()
    tick_labels = [text.get_text() for text in xticks[1]]
    pvalue = kruskal(*metric_subsets).pvalue
    if pvalue >= 0.001:
        pvalue_text = f"Kruskal-Wallis, p = {pvalue:.3f}"
    else:
        pvalue_text = f"Kruskal-Wallis, p = {pvalue:.2e}"
    return pvalue, pvalue_text, tick_labels

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(10, 3))
pvalue, pvalue_text, algs = raincloud_plots_variables(algs_df, "algorithm", "gps", ax, "General performance score")
ax.set_xlabel("Algorithm")
# ax.legend(title=f"{pvalue_text}", loc=2, bbox_to_anchor=(0, 0.2))
plt.savefig('figures/algs.svg', bbox_inches='tight')

Plots for number of views

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(7, 4))
pvalue, pvalue_text, views = raincloud_plots_variables(clusters_df, "n_clusters", "gps", ax, "General performance score")
ax.set_xlabel("No. views")
# ax.legend(title=f"{pvalue_text}", loc=2, bbox_to_anchor=(0, 1))
plt.savefig('figures/views.svg', bbox_inches='tight')

In [ ]:
mean_values = algs_df.groupby("algorithm")["gps"].mean().sort_values(ascending=False)
order = mean_values.index.tolist()

plt.figure(figsize=(10, 4))
ax = sns.boxplot(data=clusters_df, x="algorithm", y="gps", hue="n_clusters", order=order, palette="colorblind", showmeans=True)

plt.legend(title='n_clusters', title_fontsize='12')
plt.xlabel("Algorithm")
plt.ylabel("General performance score")
plt.legend(title="No. clusters", bbox_to_anchor=(1, 1))
plt.savefig('figures/clusters_by_alg.svg', bbox_inches='tight')
plt.show()

In [ ]:
cna_methyl = clusters_df[clusters_df["modalities"] == "110000"]
mean_values = algs_multi.groupby("algorithm")["gps"].mean().sort_values(ascending=False)
order = mean_values.index.tolist()

plt.figure(figsize=(10, 4))
ax = sns.barplot(data=cna_methyl, x="algorithm", y="gps", hue="n_clusters", order=order, palette="colorblind")

plt.legend(title='n_clusters', title_fontsize='12')
plt.xlabel("Algorithm")
plt.ylabel("General performance score")
plt.legend(title="No. clusters", bbox_to_anchor=(1, 1))
# plt.savefig('figures/clusters_by_alg.svg', bbox_inches='tight')
plt.show()